# CFM vs standard MLP across hidden widths

Train **100 networks per method at each width: 32, 64, and 128** (600 networks in total). Methods share the update count, learning rate, training points, evaluation grid, and seed IDs at each width. Paired seeds do not imply identical initial parameters: CFM has two inputs and `4*hidden_size+1` parameters; the standard MLP has one input and `3*hidden_size+1` parameters.

The main metric is **final grid MSE**. CFM evaluation starts from zero noise at every x. Training MSE and training time are also recorded. The batch displays progress instead of live fit charts.

Each width has its own pair of box plots with individual runs and highlighted **arithmetic means**. The summary table groups statistics by hidden width and method. The MSE axis is logarithmic; means are arithmetic means of the original MSE values.

Completed models are saved to `notebooks/outputs/cfm_vs_standard.csv`. The training cell **resumes** matching saved results, skipping completed `(hidden_size, method, seed)` combinations. Existing CSV files without a width column are read using the width in their saved configuration, preserving the earlier 32-neuron results. To start a different experiment, change `results_path`; incompatible saved training settings are rejected.

In [1]:
from pathlib import Path
import sys
import csv
import json
from dataclasses import asdict, replace
from time import perf_counter

project_root = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pyproject.toml").is_file() and (p / "notebooks" / "_shared").is_dir()
)
shared_parent = str(project_root / "notebooks")
if shared_parent not in sys.path:
    sys.path.insert(0, shared_parent)

import numpy as np
import torch
import ipywidgets as widgets
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown
from _shared import Config, Experiment
from _shared.conditional_flow_matching import FlowMatchingConfig
from _shared.flow_matching_experiment import FlowMatchingExperiment

torch.set_num_threads(1)

In [2]:
n_runs = 100  # Per method and width: 600 networks in total.
hidden_sizes = (32, 64, 128)
config = Config(
    total_steps=20_000,
    learning_rate=0.01,
    seed=42,
    n_train=256,
    n_grid=601,
    x_min=-3.0,
    x_max=3.0,
)
flow_config = FlowMatchingConfig(
    eval_sampling_steps=1,
    train_sampling_steps=None,
)
methods = ("Standard MLP", "CFM")
output_dir = project_root / "notebooks" / "outputs"
results_path = output_dir / "cfm_vs_standard.csv"
metadata_path = results_path.with_name(results_path.stem + "_config.json")
columns = ["hidden_size", "method", "seed", "train_mse", "grid_mse", "training_seconds"]


def load_saved_results():
    if not results_path.exists():
        return []
    saved_metadata = (
        json.loads(metadata_path.read_text(encoding="utf-8"))
        if metadata_path.exists()
        else {}
    )
    legacy_width = saved_metadata.get("config", {}).get("hidden_size")
    records = []
    with results_path.open(newline="", encoding="utf-8") as stream:
        for row in csv.DictReader(stream):
            width = row.get("hidden_size") or legacy_width
            if width is None:
                raise ValueError(
                    "Cannot infer the hidden width of saved results without their configuration."
                )
            records.append(
                {
                    "hidden_size": int(width),
                    "method": row["method"],
                    "seed": int(row["seed"]),
                    "train_mse": float(row["train_mse"]),
                    "grid_mse": float(row["grid_mse"]),
                    "training_seconds": float(row["training_seconds"]),
                }
            )
    return records

In [3]:
if not isinstance(n_runs, int) or n_runs < 1:
    raise ValueError("n_runs must be a positive integer.")
if not hidden_sizes or any(
    not isinstance(width, int) or width < 1 for width in hidden_sizes
):
    raise ValueError("hidden_sizes must contain positive integers.")
if len(set(hidden_sizes)) != len(hidden_sizes):
    raise ValueError("hidden_sizes must not contain duplicates.")

results = load_saved_results()
metadata = {
    "hidden_sizes": list(hidden_sizes),
    "n_runs_per_method_per_size": n_runs,
    "config": asdict(config),
    "flow_config": asdict(flow_config),
    "primary_metric": "grid_mse",
    "cfm_evaluation_initial_noise": 0,
    "torch_version": torch.__version__,
}
if results:
    if not metadata_path.exists():
        raise ValueError(
            "Saved results have no configuration. Use a new results_path for this batch."
        )
    previous = json.loads(metadata_path.read_text(encoding="utf-8"))
    old_config = {
        key: value for key, value in previous["config"].items() if key != "hidden_size"
    }
    new_config = {
        key: value for key, value in metadata["config"].items() if key != "hidden_size"
    }
    if old_config != new_config or any(
        previous.get(key) != metadata[key]
        for key in ("flow_config", "primary_metric", "cfm_evaluation_initial_noise")
    ):
        raise ValueError(
            "Saved training settings differ. Restore them or use a new results_path."
        )

output_dir.mkdir(parents=True, exist_ok=True)
# Migrate the CSV schema while preserving every completed run.
temporary_path = results_path.with_suffix(".csv.tmp")
with temporary_path.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=columns)
    writer.writeheader()
    writer.writerows(results)
temporary_path.replace(results_path)
metadata_path.write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")

finished = {(row["hidden_size"], row["method"], row["seed"]) for row in results}
tasks = [
    (width, method, config.seed + run_index)
    for width in hidden_sizes
    for run_index in range(n_runs)
    for method in methods
]
completed = widgets.IntProgress(
    value=sum(task in finished for task in tasks),
    min=0,
    max=len(tasks),
    description="Networks:",
    layout=widgets.Layout(width="100%"),
)
current = widgets.HTML()
display(widgets.VBox([completed, current]))
progress_every = max(1, config.total_steps // 100)

for width, method, seed in tasks:
    if (width, method, seed) in finished:
        continue
    run_config = replace(config, hidden_size=width, seed=seed)
    experiment = (
        Experiment(run_config)
        if method == "Standard MLP"
        else FlowMatchingExperiment(run_config, flow_config)
    )
    started = perf_counter()
    for progress in experiment.run():
        if progress.step % progress_every == 0 or progress.step == config.total_steps:
            current.value = (
                f"<b>{method}</b> | {width} neurons | run {seed - config.seed + 1}/{n_runs}"
                f" | seed {seed} | step {progress.step:,}/{config.total_steps:,}"
            )
    training_seconds = perf_counter() - started
    snapshot = experiment.evaluate_model(experiment.model, experiment.progress)
    if not np.isfinite([snapshot.train_mse, snapshot.grid_mse]).all():
        raise FloatingPointError(
            f"Nonfinite final MSE: {method}, width {width}, seed {seed}."
        )
    record = {
        "hidden_size": width,
        "method": method,
        "seed": seed,
        "train_mse": snapshot.train_mse,
        "grid_mse": snapshot.grid_mse,
        "training_seconds": training_seconds,
    }
    with results_path.open("a", newline="", encoding="utf-8") as stream:
        csv.DictWriter(stream, fieldnames=columns).writerow(record)
    results.append(record)
    finished.add((width, method, seed))
    completed.value += 1
    del experiment

completed.bar_style = "success"
current.value = f"Finished {completed.value}/{len(tasks)} networks. Results saved to <code>{results_path.name}</code>."

## Final MSE distributions by hidden width

The report groups completed runs by width and method, including partial batches. To view saved results in a fresh kernel, run the imports and configuration cells, then the CSV-loading cell below instead of the training cell. Existing 32-neuron results can be viewed before training the remaining widths.

In [4]:
# Also reads earlier single-width CSV files using their saved configuration.
results = load_saved_results()

In [5]:
if not results:
    raise ValueError(
        "No completed runs are available. Train models or load saved results first."
    )

summary = []
for width in hidden_sizes:
    for method in methods:
        values = np.array(
            [
                row["grid_mse"]
                for row in results
                if row["hidden_size"] == width
                and row["method"] == method
                and config.seed <= row["seed"] < config.seed + n_runs
            ]
        )
        if not len(values):
            continue
        summary.append(
            {
                "hidden_size": width,
                "method": method,
                "runs": len(values),
                "mean": float(values.mean()),
                "std": float(values.std(ddof=1)) if len(values) > 1 else float("nan"),
                "min": float(values.min()),
                "q25": float(np.quantile(values, 0.25)),
                "median": float(np.median(values)),
                "q75": float(np.quantile(values, 0.75)),
                "max": float(values.max()),
            }
        )
if not summary:
    raise ValueError("No saved runs match the selected hidden widths and seed range.")

table = [
    "| Hidden neurons | Method | Runs | Mean | Sample std | Min | Q25 | Median | Q75 | Max |",
    "|--:|:--|--:|--:|--:|--:|--:|--:|--:|--:|",
]
for row in summary:
    numbers = [
        f"{row[key]:.6g}"
        for key in ("mean", "std", "min", "q25", "median", "q75", "max")
    ]
    table.append(
        f"| {row['hidden_size']} | {row['method']} | {row['runs']} | "
        + " | ".join(numbers)
        + " |"
    )
display(Markdown("### Final grid MSE by hidden width\n\n" + "\n".join(table)))

### Final grid MSE by hidden width

| Hidden neurons | Method | Runs | Mean | Sample std | Min | Q25 | Median | Q75 | Max |
|--:|:--|--:|--:|--:|--:|--:|--:|--:|--:|
| 32 | Standard MLP | 100 | 0.0195202 | 0.0195702 | 0.000295755 | 0.00288547 | 0.0077071 | 0.0304163 | 0.0664809 |
| 32 | CFM | 100 | 0.00161295 | 0.00129855 | 0.000200195 | 0.000643771 | 0.00148449 | 0.00196762 | 0.00552163 |
| 64 | Standard MLP | 100 | 0.00273781 | 0.0064765 | 8.56899e-05 | 0.000382694 | 0.000961283 | 0.00191871 | 0.0397085 |
| 64 | CFM | 100 | 0.000710814 | 0.000748866 | 0.000112027 | 0.000302303 | 0.000455242 | 0.000792328 | 0.00450412 |
| 128 | Standard MLP | 100 | 0.00117418 | 0.00308153 | 4.98816e-05 | 0.00015827 | 0.000277764 | 0.000491659 | 0.0239431 |
| 128 | CFM | 100 | 0.000692122 | 0.00107245 | 7.71405e-05 | 0.000143237 | 0.000271968 | 0.000532161 | 0.0059951 |

In [6]:
report_widths = [
    width
    for width in hidden_sizes
    if any(row["hidden_size"] == width for row in summary)
]
figure = make_subplots(
    rows=1,
    cols=len(report_widths),
    shared_yaxes=True,
    subplot_titles=[f"{width} hidden neurons" for width in report_widths],
    horizontal_spacing=0.08,
)
colors = {"Standard MLP": "#2563eb", "CFM": "#16a34a"}
plot_floor = np.finfo(float).tiny
for column, width in enumerate(report_widths, start=1):
    for method in methods:
        rows = [
            row
            for row in results
            if row["hidden_size"] == width
            and row["method"] == method
            and config.seed <= row["seed"] < config.seed + n_runs
        ]
        if not rows:
            continue
        figure.add_trace(
            go.Box(
                name=method,
                legendgroup=method,
                showlegend=column == 1,
                x=[method] * len(rows),
                y=[max(row["grid_mse"], plot_floor) for row in rows],
                customdata=[[row["seed"], row["train_mse"]] for row in rows],
                boxpoints="all",
                jitter=0.35,
                pointpos=0,
                quartilemethod="linear",
                marker=dict(color=colors[method], size=5, opacity=0.55),
                line=dict(color=colors[method], width=2),
                hovertemplate=(
                    f"{method} | {width} neurons<br>Grid MSE: %{{y:.6g}}<br>Seed: %{{customdata[0]}}"
                    "<br>Train MSE: %{customdata[1]:.6g}<extra></extra>"
                ),
            ),
            row=1,
            col=column,
        )
    means = [row for row in summary if row["hidden_size"] == width]
    figure.add_trace(
        go.Scatter(
            x=[row["method"] for row in means],
            y=[max(row["mean"], plot_floor) for row in means],
            text=[f"{row['mean']:.3g}" for row in means],
            customdata=[[row["runs"]] for row in means],
            mode="markers+text",
            textposition="top center",
            textfont=dict(size=11),
            cliponaxis=False,
            marker=dict(
                symbol="diamond",
                color="#f59e0b",
                size=13,
                line=dict(color="#78350f", width=2),
            ),
            name="Arithmetic mean",
            legendgroup="mean",
            showlegend=column == 1,
            hovertemplate=(
                f"%{{x}} | {width} neurons<br>Mean MSE: %{{y:.6g}}"
                "<br>Runs: %{customdata[0]}<extra></extra>"
            ),
        ),
        row=1,
        col=column,
    )
figure.update_xaxes(
    title_text="Method", categoryorder="array", categoryarray=list(methods)
)
figure.update_yaxes(type="log")
figure.update_yaxes(title_text="Final grid MSE (lower is better)", row=1, col=1)
figure.update_layout(
    title=dict(
        text="Final grid MSE by hidden width",
        x=0,
        xanchor="left",
        y=0.98,
        yanchor="top",
        font=dict(size=20),
    ),
    template="plotly_white",
    autosize=True,
    height=650,
    margin=dict(l=80, r=35, t=100, b=130),
    legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.2, yanchor="top"),
)
figure.show(config={"responsive": True, "displayModeBar": False})